**SQL Integration**

In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import os

# 1. Filenames
dataset_file   = 'Sample Dataset for Model.csv'
sql_query_file = 'cta_analysis_queries.sql'

# 2. PostgreSQL Connection Details
db_params = {
    "host"    : "localhost",
    "port"    : 5432,
    "database": "bus_crowding",
    "user"    : "postgres",
    "password": "Nansathik@786"
}

password_encoded = quote_plus(db_params["password"])
engine = create_engine(
    f"postgresql+psycopg2://{db_params['user']}:{password_encoded}"
    f"@{db_params['host']}:{db_params['port']}/{db_params['database']}",
    connect_args={"connect_timeout": 10}
)

print(f"Active Workspace: {os.getcwd()}")

# ── STEP 1: Load CSV and Upload to PostgreSQL ─────────────────
if os.path.exists(dataset_file):
    print(f"\nFound '{dataset_file}'. Loading...")
    df_raw = pd.read_csv(dataset_file)
    df_raw.columns = ['station_id', 'stationname', 'date', 'daytype', 'rides']
    print(f"Rows    : {len(df_raw):,}")
    print(f"Columns : {list(df_raw.columns)}")

    try:
        with engine.connect() as conn:
            conn.execute(text("DROP TABLE IF EXISTS daily_ridership;"))
            conn.execute(text("""
                CREATE TABLE daily_ridership (
                    station_id  INTEGER,
                    stationname VARCHAR(255),
                    date        DATE,
                    daytype     VARCHAR(5),
                    rides       INTEGER
                );
            """))
            conn.commit()
            print("Table 'daily_ridership' created.")

        df_raw.to_sql(
            'daily_ridership', engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print(f"Upload successful — {len(df_raw):,} rows inserted.")

    except Exception as e:
        print(f"Upload error: {e}")
else:
    print(f"\nError: '{dataset_file}' not found in {os.getcwd()}")

# ── STEP 2: Run SQL Queries and Save CSVs ─────────────────────
if os.path.exists(sql_query_file):
    print("\nRunning SQL Analysis Queries...")
    try:
        queries = {
            "top_10_stations.csv": """
                SELECT stationname,
                       SUM(rides) AS total_rides
                FROM   daily_ridership
                GROUP  BY stationname
                ORDER  BY total_rides DESC
                LIMIT  10
            """,
            "avg_rides_by_daytype.csv": """
                SELECT
                    CASE daytype
                        WHEN 'W' THEN 'Weekday'
                        WHEN 'A' THEN 'Saturday'
                        WHEN 'U' THEN 'Sunday/Holiday'
                    END AS "Day Category",
                    ROUND(AVG(rides)::NUMERIC, 2) AS average_rides
                FROM daily_ridership
                GROUP BY daytype
            """
        }

        for filename, query in queries.items():
            with engine.connect() as conn:
                output_df = pd.read_sql(text(query), conn)
            output_df.to_csv(filename, index=False)
            print(f"\nFile Generated : {filename}")
            print(output_df.to_string(index=False))

        print("\n" + "="*50)
        print("  PIPELINE COMPLETE")
        print("="*50)
        print("  top_10_stations.csv       — saved")
        print("  avg_rides_by_daytype.csv  — saved")
        print("="*50)
        print("Continue with model training.")

    except Exception as e:
        print(f"Query error: {e}")
else:
    print(f"\nError: '{sql_query_file}' not found.")

Active Workspace: c:\Crowd Forecast

Found 'Sample Dataset for Model.csv'. Loading...
Rows    : 962,546
Columns : ['station_id', 'stationname', 'date', 'daytype', 'rides']
Table 'daily_ridership' created.
Upload successful — 962,546 rows inserted.

Running SQL Analysis Queries...

File Generated : top_10_stations.csv
       stationname  total_rides
        Clark/Lake     94799999
        Lake/State     94264858
     Chicago/State     87346160
     95th/Dan Ryan     70781223
Belmont-North Main     70676132
         Fullerton     69360576
       Grand/State     64805660
    O'Hare Airport     62189238
     Jackson/State     59499616
         Roosevelt     57732881

File Generated : avg_rides_by_daytype.csv
  Day Category  average_rides
      Saturday        2274.90
Sunday/Holiday        1649.81
       Weekday        3911.05

  PIPELINE COMPLETE
  top_10_stations.csv       — saved
  avg_rides_by_daytype.csv  — saved
Continue with model training.


**Data Preprocessing**

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import datetime

In [3]:
df = pd.read_csv('Sample Dataset for model.csv')

In [4]:
#Convert 'date' column to datetime format
df['date'] = pd.to_datetime(df['date'])

In [5]:
#Extract Time Features (Machine Learning needs numbers, not date strings)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek # Monday=0, Sunday=6

In [6]:
#Convert 'daytype' to numerical values
# W (Weekday) = 1, A (Saturday) = 2, U (Sunday/Holiday) = 3
day_map = {'W': 1, 'A': 2, 'U': 3}
df['day_type_num'] = df['daytype'].map(day_map)

In [7]:
# Display all unique stations in the dataset
stations_info = df[['station_id', 'stationname']].drop_duplicates().sort_values('stationname')
print(f'Total unique stations in dataset: {len(stations_info)}')
print('\nAll Stations:')
print(stations_info.to_string(index=False))

Total unique stations in dataset: 148

All Stations:
 station_id              stationname
      40830                     18th
      41120       35-Bronzeville-IIT
      40120              35th/Archer
      41270                     43rd
      41230            47th-Dan Ryan
      41080      47th-South Elevated
      40130                     51st
      40580              54th/Cermak
      40910            63rd-Dan Ryan
      40990                     69th
      40240                     79th
      41430                     87th
      40450            95th/Dan Ryan
      40680             Adams/Wabash
      41440            Addison-Brown
      41420       Addison-North Main
      41240           Addison-O'Hare
      41200                   Argyle
      40660                 Armitage
      40170             Ashland-Lake
      41060           Ashland-Orange
      40290             Ashland/63rd
      40010       Austin-Forest Park
      41260              Austin-Lake
      41320       Belm

In [8]:
print("Preprocessing Complete. Dataset shape:", df.shape)
print("Top 5 rows of processed data:")
print(df.head())

Preprocessing Complete. Dataset shape: (962546, 10)
Top 5 rows of processed data:
   station_id       stationname       date daytype  rides  year  month  day  \
0       40350       UIC-Halsted 2001-01-01       U    273  2001      1    1   
1       41130    Halsted-Orange 2001-01-01       U    306  2001      1    1   
2       40760         Granville 2001-01-01       U   1059  2001      1    1   
3       40070  Jackson/Dearborn 2001-01-01       U    649  2001      1    1   
4       40090       Damen-Brown 2001-01-01       U    411  2001      1    1   

   day_of_week  day_type_num  
0            0             3  
1            0             3  
2            0             3  
3            0             3  
4            0             3  


**Model Training**

In [9]:
# Train a separate Random Forest model for every station
# This allows accurate, station-specific predictions

station_models = {}  # Dictionary: station_id -> trained model
station_mae = {}     # Dictionary: station_id -> MAE score

feature_cols = ['year', 'month', 'day', 'day_of_week', 'day_type_num']
unique_stations = df['station_id'].unique()

print(f'Training models for {len(unique_stations)} stations...')
print('-' * 50)

for station_id in unique_stations:
    station_data = df[df['station_id'] == station_id].sort_values('date')
    X = station_data[feature_cols]
    y = station_data['rides']

    if len(station_data) < 50 or station_data['rides'].mean() == 0:
       # Use global average for sparse stations
            avg_rides = int(station_data['rides'].mean())
            station_models[station_id] = None
            station_mae[station_id] = 0
            continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    station_models[station_id] = model
    station_mae[station_id] = mae

print(f'Done! Trained {len(station_models)} station models successfully.')
print(f'Average MAE across all stations: {np.mean(list(station_mae.values())):.2f} passengers')

Training models for 147 stations...
--------------------------------------------------
Done! Trained 147 station models successfully.
Average MAE across all stations: 193.61 passengers


**Predicting Future Crowd**

In [10]:
import datetime

# --- INTERACTIVE CROWD PREDICTION SYSTEM (ALL STATIONS) ---

print('--- CTA Ridership Prediction Tool (All Stations) ---')

try:
    print('\nPlease enter the date details below:')
    input_year  = int(input('Enter Year  (e.g., 2026): '))
    input_month = int(input('Enter Month (1-12)      : '))
    input_day   = int(input('Enter Day   (1-31)      : '))

    target_date  = datetime.date(input_year, input_month, input_day)
    day_of_week  = target_date.weekday()

    if day_of_week == 5:
        day_type_num = 2   # Saturday
        day_category = 'Saturday'
    elif day_of_week == 6:
        day_type_num = 3   # Sunday/Holiday
        day_category = 'Sunday/Holiday'
    else:
        day_type_num = 1   # Weekday
        day_category = 'Weekday'

    future_features = [[input_year, input_month, input_day, day_of_week, day_type_num]]

    # Predict for every station that has a trained model
    results = []
    for station_id, model in station_models.items():
        model = station_models[station_id]
        if model is None:
            station_data = df[df['station_id'] == station_id]
            predicted_rides = int(station_data['rides'].mean())
        else:
            predicted_rides = int(model.predict(future_features)[0])
        station_name = df.loc[df['station_id'] == station_id, 'stationname'].iloc[0]
        results.append({
            'Station ID'            : station_id,
            'Station Name'          : station_name,
            'Estimated Passengers'  : predicted_rides
        })

    results_df = pd.DataFrame(results).sort_values(
        'Estimated Passengers', ascending=False
    ).reset_index(drop=True)
    results_df.index += 1  # Start rank from 1

    total_passengers = results_df['Estimated Passengers'].sum()

    # --- Display Results ---
    print('\n' + '='*60)
    print(f"  PREDICTION FOR : {target_date.strftime('%B %d, %Y')}")
    print(f"  Day Name       : {target_date.strftime('%A')}")
    print(f"  Day Category   : {day_category}")
    print(f"  Stations       : {len(results_df)}")
    print('='*60)
    print()
    print('ESTIMATED PASSENGERS PER STATION (ranked by ridership):')
    print('-'*60)
    print(results_df.to_string())
    print('-'*60)
    print(f'TOTAL ESTIMATED PASSENGERS (all stations): {total_passengers:,}')
    print('='*60)

except ValueError:
    print('\n[Error]: Please enter valid numbers for Year, Month, and Day.')
except Exception as e:
    print(f'\n[Error]: An unexpected error occurred: {e}')

--- CTA Ridership Prediction Tool (All Stations) ---

Please enter the date details below:


c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\util


  PREDICTION FOR : October 07, 2037
  Day Name       : Wednesday
  Day Category   : Weekday
  Stations       : 147

ESTIMATED PASSENGERS PER STATION (ranked by ridership):
------------------------------------------------------------
     Station ID              Station Name  Estimated Passengers
1         40380                Clark/Lake                 23211
2         41660                Lake/State                 23041
3         41220                 Fullerton                 15739
4         41450             Chicago/State                 14820
5         40370       Washington/Dearborn                 14710
6         40260                State/Lake                 14292
7         41320        Belmont-North Main                 13206
8         40890            O'Hare Airport                 11503
9         41400                 Roosevelt                 11484
10        40560             Jackson/State                 11333
11        40330               Grand/State                 1132

**Exporting Results for Power BI**

In [11]:
import pandas as pd
import datetime
import os

# ── PART 1: Generate Full Historical Forecast File ────────────
print("Generating final_forecast_results.csv for all stations...")

feature_cols = ['year', 'month', 'day', 'day_of_week', 'day_type_num']
all_predicted = []

for station_id, model in station_models.items():
    station_data = df[df['station_id'] == station_id].copy()

    if model is None:
        station_data['predicted_rides'] = int(station_data['rides'].mean())
    else:
        station_data['predicted_rides'] = model.predict(
            station_data[feature_cols]
        ).astype(int)

    all_predicted.append(station_data)

# Combine all stations
full_forecast_df = pd.concat(all_predicted, ignore_index=True)

# Rename to match required column names
full_forecast_df = full_forecast_df.rename(columns={
    'rides'          : 'Actual Passengers',
    'predicted_rides': 'Forecasted Crowd'
})

# Add Day Name column
full_forecast_df['date'] = pd.to_datetime(full_forecast_df['date'])
full_forecast_df['Day Name'] = full_forecast_df['date'].dt.day_name()

# Verify columns before saving
required_cols = ['station_id', 'stationname', 'date', 'daytype',
                 'Actual Passengers', 'year', 'month', 'day',
                 'day_of_week', 'day_type_num', 'Forecasted Crowd', 'Day Name']

missing = [c for c in required_cols if c not in full_forecast_df.columns]
if missing:
    print(f"Warning — missing columns: {missing}")
else:
    print("All required columns present.")

# Save
full_forecast_df.to_csv('final_forecast_results.csv', index=False)

print(f"\nFILE 1 SAVED: final_forecast_results.csv")
print(f"  Rows             : {len(full_forecast_df):,}")
print(f"  Stations covered : {full_forecast_df['station_id'].nunique()}")
print(f"  Columns          : {list(full_forecast_df.columns)}")

# ── PART 2: Generate Specific Date Forecast Report ────────────
print(f"\nGenerating forecast_report for {input_year}-{input_month}-{input_day}...")

specific_date_results = []
stations_list = df[['station_id', 'stationname']].drop_duplicates()
input_features = [[input_year, input_month, input_day, day_of_week, day_type_num]]

for index, row in stations_list.iterrows():
    sid   = row['station_id']
    sname = row['stationname']
    model_for_station = station_models.get(sid)

    if model_for_station is None:
        predicted_val = int(df[df['station_id'] == sid]['rides'].mean())
    else:
        predicted_val = int(model_for_station.predict(input_features)[0])

    specific_date_results.append({
        'Station ID'          : sid,
        'Station Name'        : sname,
        'Estimated Passengers': predicted_val
    })

report_df = pd.DataFrame(specific_date_results).sort_values(
    'Estimated Passengers', ascending=False
).reset_index(drop=True)
report_df.index += 1

report_filename = f'forecast_report_{input_year}_{input_month}_{input_day}.csv'
report_df.to_csv(report_filename, index=False)

print(f"\nFILE 2 SAVED: {report_filename}")
print(f"  Stations : {len(report_df)}")
print(f"  Columns  : {list(report_df.columns)}")
print(report_df.head(10).to_string())

print("\n" + "="*55)
print("  EXPORT COMPLETE — 2 FILES READY FOR POWER BI")
print("="*55)
print(f"  1. final_forecast_results.csv")
print(f"  2. {report_filename}")
print("="*55)

Generating final_forecast_results.csv for all stations...
All required columns present.

FILE 1 SAVED: final_forecast_results.csv
  Rows             : 962,546
  Stations covered : 147
  Columns          : ['station_id', 'stationname', 'date', 'daytype', 'Actual Passengers', 'year', 'month', 'day', 'day_of_week', 'day_type_num', 'Forecasted Crowd', 'Day Name']

Generating forecast_report for 2037-10-7...


c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\util


FILE 2 SAVED: forecast_report_2037_10_7.csv
  Stations : 148
  Columns  : ['Station ID', 'Station Name', 'Estimated Passengers']
    Station ID         Station Name  Estimated Passengers
1        40380           Clark/Lake                 23211
2        41660           Lake/State                 23041
3        41220            Fullerton                 15739
4        41450        Chicago/State                 14820
5        40370  Washington/Dearborn                 14710
6        40260           State/Lake                 14292
7        41320   Belmont-North Main                 13206
8        40890       O'Hare Airport                 11503
9        41400            Roosevelt                 11484
10       40560        Jackson/State                 11333

  EXPORT COMPLETE — 2 FILES READY FOR POWER BI
  1. final_forecast_results.csv
  2. forecast_report_2037_10_7.csv


**Power BI**

In [12]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os

print("Storing all 4 output files into PostgreSQL database...")

# Upload each file as a table
tables = {
    'top_10_stations'       : 'top_10_stations.csv',
    'avg_rides_by_daytype'  : 'avg_rides_by_daytype.csv',
    'final_forecast_results': 'final_forecast_results.csv',
    'forecast_report'       : f'forecast_report_{input_year}_{input_month}_{input_day}.csv'
}

for table_name, filename in tables.items():
    if os.path.exists(filename):
        df_temp = pd.read_csv(filename)
        df_temp.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"Table saved : {table_name} ({len(df_temp):,} rows)")
    else:
        print(f"Warning     : {filename} not found — skipping.")

# Verify all tables in database
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))
    tables_in_db = [row[0] for row in result]

print("\n" + "="*55)
print("  DATABASE UPDATE COMPLETE")
print("="*55)
print(f"  Host     : {db_params['host']}")
print(f"  Database : {db_params['database']}")
print(f"  Tables   : {tables_in_db}")
print("="*55)
print("Now connect Power BI to this PostgreSQL database.")

Storing all 4 output files into PostgreSQL database...
Table saved : top_10_stations (10 rows)
Table saved : avg_rides_by_daytype (3 rows)
Table saved : final_forecast_results (962,546 rows)
Table saved : forecast_report (148 rows)

  DATABASE UPDATE COMPLETE
  Host     : localhost
  Database : bus_crowding
  Tables   : ['avg_rides_by_daytype', 'daily_ridership', 'final_forecast_results', 'forecast_report', 'top_10_stations']
Now connect Power BI to this PostgreSQL database.
